# DataPulse â€” Exploratory Data Analysis

All data is pulled from the **Gold layer** (`gold.*` BigQuery tables).  
Never query Bronze or Silver directly from notebooks.

**Key narrative threads:**
1. The dataset ends in 2018 â€” every customer looks inactive against today's date; a forward-window design is required for ML labels.
2. ~97% of customers ordered exactly once â€” the ML model is a **repeat-purchase propensity model** (positive class = the small minority of returners).

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv
from google.cloud import bigquery
warnings.filterwarnings('ignore')


load_dotenv()
PROJECT = os.environ['GCP_PROJECT_ID']
client = bigquery.Client(project=PROJECT)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print(f'Connected to project: {PROJECT}')

In [ ]:
def q(sql):
    return client.query(sql).to_dataframe()

# Load gold tables
orders     = q(f'SELECT * FROM `{PROJECT}.gold.fct_orders`')
customers  = q(f'SELECT * FROM `{PROJECT}.gold.dim_customers`')
order_items= q(f'SELECT * FROM `{PROJECT}.gold.fct_order_items`')
cohorts    = q(f'SELECT * FROM `{PROJECT}.gold.customer_cohorts`')
products   = q(f'SELECT * FROM `{PROJECT}.gold.dim_products`')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

print(f'Orders:     {len(orders):,}')
print(f'Customers:  {len(customers):,}')
print(f'Order items:{len(order_items):,}')
print(f'Cohorts:    {len(cohorts):,}')

## 1. Order Volume Over Time

Shows the dataset spans Sep 2016 â€“ Aug 2018 with strong growth.  
The sharp dropoff at the end is truncated data, not a real demand collapse.

In [ ]:
monthly = (
    orders.groupby('order_month')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'order_count'})
)
monthly['order_month_dt'] = monthly['order_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly['order_month_dt'], monthly['order_count'], width=20, color='steelblue')
ax.set_title('Monthly Order Volume (Sep 2016 â€“ Aug 2018)')
ax.set_xlabel('Month')
ax.set_ylabel('Order Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
plt.show()

## 2. Data End-Date Narrative

**Why this matters for ML labeling:** `current_date()` in dbt would make every customer's `recency_days` ~2,800 days, and every customer would appear inactive. The repeat-purchase propensity model sidesteps this entirely by using a forward-window label (`will_repeat`) computed from a snapshot date T inside the dataset â€” immune to the frozen-clock problem.

In [ ]:
last_order = orders['order_purchase_timestamp'].max()
first_order = orders['order_purchase_timestamp'].min()
print(f'Dataset spans: {first_order.date()} â†’ {last_order.date()}')
print(f'Days from last order to today (2026-06-26): {(pd.Timestamp("2026-06-26", tz="UTC") - last_order).days:,}')

fig, ax = plt.subplots(figsize=(12, 3))
customers['last_order_date'] = pd.to_datetime(customers['last_order_date'])
ax.hist(customers['last_order_date'], bins=50, color='coral', edgecolor='white')
ax.axvline(last_order, color='navy', linestyle='--', label=f'Max purchase date: {last_order.date()}')
ax.set_title('Distribution of Customer Last Order Dates (all end before Oct 2018)')
ax.set_xlabel('Last Order Date')
ax.set_ylabel('Customer Count')
ax.legend()
fig.tight_layout()
plt.show()

## 3. One-Time Buyer Reality

~97% of customers placed exactly one order. This motivates the **repeat-purchase propensity** framing:  
the positive class (`will_repeat = true`) is the small, interesting minority of returners (~1â€“3%).  
`scale_pos_weight` in XGBoost up-weights this minority so the model doesn't ignore it.

In [ ]:
freq_counts = customers['frequency'].value_counts().sort_index().head(10)
pct_one_time = (customers['frequency'] == 1).mean() * 100
print(f'One-time buyers: {pct_one_time:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(freq_counts.index.astype(str), freq_counts.values, color='steelblue')
ax.set_title(f'Order Frequency Distribution ({pct_one_time:.1f}% of customers ordered exactly once)')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Customer Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
plt.show()

## 4. Revenue by State

In [ ]:
revenue_state = (
    orders.groupby('customer_state')['total_payment_value']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 5))
revenue_state.plot.bar(ax=ax, color='teal')
ax.set_title('Total Revenue by State (Top 15)')
ax.set_xlabel('State')
ax.set_ylabel('Revenue (BRL)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
fig.tight_layout()
plt.show()

## 5. Review Score Distribution

In [ ]:
review_counts = orders['review_score'].value_counts().sort_index().dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(review_counts.index, review_counts.values, color=['#d9534f','#f0ad4e','#f0ad4e','#5cb85c','#5bc0de'][:len(review_counts)])
ax.set_title('Review Score Distribution (1â€“5 stars)')
ax.set_xlabel('Review Score')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(ax.patches, review_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
plt.show()

## 6. Delivery Time & SLA Performance

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].dropna(subset=['days_to_deliver'])
on_time_pct = delivered['delivered_on_time'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(delivered['days_to_deliver'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Days to Deliver (delivered orders)')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')
axes[0].axvline(delivered['days_to_deliver'].median(), color='red', linestyle='--',
                label=f'Median: {delivered["days_to_deliver"].median():.0f} days')
axes[0].legend()

axes[1].pie(
    [on_time_pct, 100 - on_time_pct],
    labels=[f'On Time ({on_time_pct:.1f}%)', f'Late ({100-on_time_pct:.1f}%)'],
    colors=['#5cb85c', '#d9534f'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[1].set_title('SLA Compliance (vs. estimated delivery date)')

fig.tight_layout()
plt.show()

## 7. Payment Method Mix

In [ ]:
payment_mix = {
    'Credit Card': customers['used_credit_card'].sum(),
    'Boleto':      customers['used_boleto'].sum(),
    'Voucher':     customers['used_voucher'].sum(),
}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(payment_mix.keys(), payment_mix.values(), color=['#337ab7','#f0ad4e','#5cb85c'])
ax.set_title('Payment Method Usage (unique customers)')
ax.set_ylabel('Customers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
fig.tight_layout()
plt.show()

## 8. Category Revenue Ranking

In [ ]:
cat_revenue = (
    order_items
    .groupby('product_category_name_english')['price']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 5))
cat_revenue.plot.barh(ax=ax, color='teal')
ax.set_title('Revenue by Product Category (Top 15)')
ax.set_xlabel('Revenue (BRL)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
fig.tight_layout()
plt.show()

## 9. Cohort Retention Heatmap

In [ ]:
cohorts['cohort_month'] = pd.to_datetime(cohorts['cohort_month'])
cohort_pivot = cohorts.pivot_table(
    index='cohort_month', columns='months_since_first', values='active_customers'
).astype(float)  # convert pd.NA â†’ np.nan so seaborn can render it
# Normalise by cohort size (month 0)
cohort_size = cohort_pivot[0]
cohort_pct = cohort_pivot.divide(cohort_size, axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    cohort_pct,
    ax=ax,
    cmap='YlOrRd_r',
    fmt='.0f',
    annot=True,
    linewidths=0.3,
    cbar_kws={'label': '% of cohort still active'},
    vmin=0,
    vmax=100,
)
ax.set_title('Cohort Retention Rate (% of cohort placing another order)')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort Month')
ax.set_yticklabels(
    [t.strftime('%Y-%m') for t in cohort_pivot.index], rotation=0
)
fig.tight_layout()
plt.show()

## 10. RFM Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(customers['recency_days'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Recency (days since last order, dataset-relative)')
axes[0].set_xlabel('Recency Days')

axes[1].hist(np.log1p(customers['monetary']), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Monetary (log-scale BRL)')
axes[1].set_xlabel('log(1 + Total Spend)')

axes[2].hist(customers['avg_review_score'].dropna(), bins=20, color='coral', edgecolor='white')
axes[2].set_title('Avg Review Score per Customer')
axes[2].set_xlabel('Average Score (1â€“5)')

for ax in axes:
    ax.set_ylabel('Customers')

fig.tight_layout()
plt.show()

## 11. Numeric Feature Correlation Heatmap

In [ ]:
numeric_cols = [
    'frequency', 'monetary', 'avg_order_value',
    'recency_days', 'avg_review_score', 'max_installments_used',
]
corr = customers[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, ax=ax, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5,
)
ax.set_title('Feature Correlation Matrix (Gold dim_customers)')
fig.tight_layout()
plt.show()

## Summary

| Finding | Implication |
|---------|-------------|
| Dataset ends Sep 2018 | Repeat-purchase propensity model uses forward-window label (`will_repeat`), not `current_date()` |
| ~97% one-time buyers | Model positive class is a small minority (~1â€“3% returners); `scale_pos_weight` required |
| SÃ£o Paulo dominates revenue | Geographic feature (`customer_state`) is informative for segmentation |
| 5-star reviews dominate | Review score signal may be weak; `has_review` flag adds information |
| Credit card â‰« boleto/voucher | Payment method features add marginal signal to propensity model |